# 00 — Настройка окружения

**Этап 1 задания на ВКР:** настройка окружения, скачивание данных и моделей.

Шаги:

1. Зависимости
2. Проверка GPU
3. Google Drive + модели (Qwen2.5-1.5B, PRM-7B)
4. Парсинг — установка оригинальных пакетов
5. Определение функций
6. Тесты + верификация
7. Сохранение parsing.py
8. Итог


## 1. Зависимости

In [1]:
!pip install -q torch transformers peft trl datasets accelerate pyyaml scipy huggingface_hub


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 678.0/678.0 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 75.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 70.9 MB/s eta 0:00:00


## 2. Проверка GPU

In [2]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name}, {gpu_mem:.1f} GB')
else:
    raise RuntimeError('GPU не доступен. Среда выполнения -> Сменить среду -> GPU')


GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, 102.0 GB


##3.1 Google Drive

In [3]:
from google.colab import drive
import os

drive.mount('/content/drive')

MODEL_ROOT = '/content/drive/MyDrive/vkr_models'
DATA_ROOT = '/content/drive/MyDrive/vkr_data'
RESULTS_ROOT = '/content/drive/MyDrive/vkr_results'

os.makedirs(MODEL_ROOT, exist_ok=True)
os.makedirs(DATA_ROOT, exist_ok=True)
os.makedirs(RESULTS_ROOT, exist_ok=True)

print(f'Модели:     {MODEL_ROOT}')
print(f'Данные:     {DATA_ROOT}')
print(f'Результаты: {RESULTS_ROOT}')

Mounted at /content/drive
Модели:     /content/drive/MyDrive/vkr_models
Данные:     /content/drive/MyDrive/vkr_data
Результаты: /content/drive/MyDrive/vkr_results


## 3.2 Скачивание Qwen2.5-1.5B

Целевая модель для SFT и GRPO.  
https://huggingface.co/Qwen/Qwen2.5-1.5B

In [4]:
from huggingface_hub import snapshot_download, model_info

TARGET_MODEL = 'Qwen/Qwen2.5-1.5B'
TARGET_DIR = f'{MODEL_ROOT}/Qwen2.5-1.5B'

snapshot_download(TARGET_MODEL, local_dir=TARGET_DIR)
assert os.path.exists(TARGET_DIR), f'Модель не скачалась: {TARGET_DIR}'

target_sha = model_info(TARGET_MODEL).sha
print(f'Модель: {TARGET_MODEL}')
print(f'Revision: {target_sha}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

Модель: Qwen/Qwen2.5-1.5B
Revision: 8faed761d45a263340a0528343f099c05c9a4323


## 3.3 Скачивание Qwen2.5-Math-PRM-7B

Process Reward Model для inference-стратегий (этап 3).  
~14 GB. Если диска не хватает — пропустить, скачать на этапе 3.  
https://huggingface.co/Qwen/Qwen2.5-Math-PRM-7B

In [5]:
PRM_MODEL = 'Qwen/Qwen2.5-Math-PRM-7B'
PRM_DIR = f'{MODEL_ROOT}/Qwen2.5-Math-PRM-7B'

snapshot_download(PRM_MODEL, local_dir=PRM_DIR)
assert os.path.exists(PRM_DIR), f'Модель не скачалась: {PRM_DIR}'

prm_sha = model_info(PRM_MODEL).sha
print(f'Модель: {PRM_MODEL}')
print(f'Revision: {prm_sha}')

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Модель: Qwen/Qwen2.5-Math-PRM-7B
Revision: 0610740060112df12585d00a1c5f4624d2f59051


In [6]:
# проверка
for name, path in [('Qwen2.5-1.5B', TARGET_DIR), ('PRM-7B', PRM_DIR)]:
    size_gb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fns in os.walk(path) for f in fns
    ) / 1e9
    print(f'{name}: {size_gb:.1f} GB в {path}')

Qwen2.5-1.5B: 3.1 GB в /content/drive/MyDrive/vkr_models/Qwen2.5-1.5B
PRM-7B: 15.3 GB в /content/drive/MyDrive/vkr_models/Qwen2.5-Math-PRM-7B


## 4. Парсинг ответов

Устанавливаем оригинальные evaluation-пакеты авторов бенчмарков:
- GSM8K: `openai/grade-school-math` (Cobbe et al., 2021), MIT
- MATH: `hendrycks/math` (Hendrycks et al., 2021), MIT

Из них берём стандартные функции сравнения ответов.  
Функции извлечения (`\boxed{}`, `####`) — общепринятый код оттуда же.  
После тестов и верификации - сохраняем в `parsing.py`.

In [7]:
# Оригинальные пакеты авторов бенчмарков
# пакет MATH
!pip install -q git+https://github.com/hendrycks/math.git

# GSM8K пакет берём напрямую из исходника: https://github.com/openai/grade-school-math/blob/master/grade_school_math/dataset.py
# ANS_RE = re.compile(r"#### (\-?[0-9\.\,]+)")

import math_equivalence
print(f'math_equivalence.is_equiv доступен: {hasattr(math_equivalence, "is_equiv")}')

  Preparing metadata (setup.py) ... done
math_equivalence.is_equiv доступен: True


In [8]:
import re
import numpy as np
import math_equivalence

# GSM8K: regex из openai/grade-school-math/dataset.py
# Верифицировано: https://github.com/openai/grade-school-math/blob/master/grade_school_math/dataset.py
ANS_RE = re.compile(r"#### (\-?[0-9\.\,]+)")

print(f'GSM8K regex: {ANS_RE.pattern}')
print(f'math_equivalence.is_equiv доступен: {hasattr(math_equivalence, "is_equiv")}')
print('n\Оригинальные пакеты импортированы')

GSM8K regex: #### (\-?[0-9\.\,]+)
math_equivalence.is_equiv доступен: True
n\Оригинальные пакеты импортированы


## 5. Определение функций

GSM8K: обёртка над оригинальным `extract_answer` (возвращаем `None` вместо `"[invalid]"`).

MATH: `is_equiv` — напрямую из `hendrycks/math`.  
Извлечение `\boxed{}` — стандартный код из того же репозитория (функция `last_boxed_only_string`).

Метрики (`accuracy`, `bootstrap_ci`) — наши.

In [9]:
# GSM8K: извлечение числового ответа после маркера ####
#
# Обёртка над openai/grade-school-math/dataset.py:extract_answer.
# Единственное отличие: возвращает None вместо "[invalid]".
# =============================================================

def extract_answer_gsm8k(text):
    '''Находит #### в тексте, возвращает число после него.
    Убирает запятые (1,000 -> 1000). Возвращает строку или None.
    Источник: openai/grade-school-math/dataset.py:extract_answer'''
    match = ANS_RE.search(text)
    if match:
        return match.group(1).strip().replace(",", "")
    return None


def is_correct_gsm8k(prediction, target):
    '''GSM8K: строковое сравнение после strip.'''
    if prediction is None:
        return False
    return prediction.strip() == target.strip()


# =============================================================
# MATH: извлечение ответа из \boxed{...}
#
# Источник: hendrycks/math/modeling/math_equivalence.py
# Копия в: stanford-crfm/helm, EleutherAI/lm-evaluation-harness
# =============================================================

def last_boxed_only_string(text):
    '''Находит последний \\boxed{...} или \\fbox{...}.
    Считает вложенные скобки. Возвращает подстроку или None.'''
    idx = text.rfind('\\boxed')
    if idx < 0:
        idx = text.rfind('\\fbox')
        if idx < 0:
            return None
    i = idx
    depth = 0
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[idx : i + 1]
        i += 1
    return None


def remove_boxed(text):
    '''Убирает \\boxed{} обёртку: \\boxed{42} -> 42'''
    left = '\\boxed{'
    try:
        assert text[:len(left)] == left
        assert text[-1] == '}'
        return text[len(left):-1]
    except Exception:
        return None


def extract_answer_math(text):
    '''Извлекает ответ из последнего \\boxed{...} в тексте.'''
    boxed = last_boxed_only_string(text)
    if boxed is None:
        return None
    return remove_boxed(boxed)


# =============================================================
# MATH: сравнение ответов — напрямую из hendrycks/math
# =============================================================

is_equiv = math_equivalence.is_equiv


# =============================================================
# Метрики
# =============================================================

def accuracy(correct):
    '''Доля правильных.'''
    if len(correct) == 0:
        return 0.0
    return sum(correct) / len(correct)


def bootstrap_ci(correct, n_bootstrap=10000, confidence=0.95, seed=42):
    '''Bootstrap 95% CI для accuracy. Возвращает (mean, lower, upper).'''
    rng = np.random.RandomState(seed)
    arr = np.array(correct, dtype=float)
    n = len(arr)
    if n == 0:
        return (0.0, 0.0, 0.0)
    means = np.array([
        rng.choice(arr, size=n, replace=True).mean()
        for _ in range(n_bootstrap)
    ])
    alpha = (1 - confidence) / 2
    lower = float(np.percentile(means, 100 * alpha))
    upper = float(np.percentile(means, 100 * (1 - alpha)))
    return (float(arr.mean()), lower, upper)


print('Все функции определены')

Все функции определены


## 6. Тесты + верификация против оригиналов

In [10]:
# --- GSM8K: извлечение ---
assert extract_answer_gsm8k('решение...\n#### 42') == '42'
assert extract_answer_gsm8k('#### 595') == '595'
assert extract_answer_gsm8k('#### 1,000') == '1000'
assert extract_answer_gsm8k('#### -3.14') == '-3.14'
assert extract_answer_gsm8k('нет маркера') is None
assert extract_answer_gsm8k('') is None
print('GSM8K extraction: 6/6')

# --- MATH: извлечение ---
assert extract_answer_math(r'ответ \boxed{42}') == '42'
assert extract_answer_math(r'\boxed{\frac{1}{4}}') == r'\frac{1}{4}'
assert extract_answer_math(r'\boxed{-\frac{2}{3}}') == r'-\frac{2}{3}'
assert extract_answer_math(r'первый \boxed{wrong} потом \boxed{24}') == '24'
assert extract_answer_math('нет boxed') is None
# Вложенные скобки глубиной 3+
assert extract_answer_math(r'\boxed{\frac{1}{\sqrt{2}}}') == r'\frac{1}{\sqrt{2}}'
assert extract_answer_math(r'\boxed{}') == ''
assert extract_answer_math(r'первый \boxed{a} затем \boxed{\frac{x^{2}+1}{3}}') == r'\frac{x^{2}+1}{3}'
print('MATH extraction: 8/8')

# --- MATH: is_equiv (из hendrycks/math) ---
assert is_equiv(r'\frac{1}{2}', '0.5') == True
assert is_equiv(r'\frac12', r'\frac{1}{2}') == True
assert is_equiv(r'\sqrt3', r'\sqrt{3}') == True
assert is_equiv(r'\dfrac{1}{2}', r'\frac{1}{2}') == True
assert is_equiv(r'5 \text{ cm}', '5') == True
assert is_equiv('42', '42') == True
assert is_equiv('42', '43') == False
assert is_equiv(None, '42') == False
print('MATH is_equiv: 8/8')

# --- GSM8K: сравнение ---
assert is_correct_gsm8k('42', '42') == True
assert is_correct_gsm8k(None, '42') == False
print('GSM8K comparison: 2/2')

# --- Метрики ---
assert accuracy([True, True, False]) == 2/3
mean, lo, hi = bootstrap_ci([True, True, False, True, False])
assert 0 <= lo <= mean <= hi <= 1
print(f'Metrics: mean={mean:.2f}, CI=[{lo:.3f}, {hi:.3f}]')

print()
print(f'Тесты пройдены: 26/26')


# Верификация:

# GSM8K: верификация regex
# Оригинал из openai/grade-school-math/dataset.py:
# ANS_RE = re.compile(r"#### (\-?[0-9\.\,]+)")
assert ANS_RE.pattern == r"#### (\-?[0-9\.\,]+)", 'GSM8K regex не совпадает с оригиналом'
print('GSM8K верификация: regex совпадает с оригиналом')

# MATH: наш is_equiv — это math_equivalence.is_equiv напрямую,
# но проверяем для уверенности
math_cases = [
    (r'\frac{1}{2}', '0.5', True),
    (r'\frac12', r'\frac{1}{2}', True),
    (r'\sqrt3', r'\sqrt{3}', True),
    (r'\dfrac{1}{2}', r'\frac{1}{2}', True),
    (r'5 \text{ cm}', '5', True),
    ('42', '42', True),
    ('42', '43', False),
    (None, '42', False),
    (None, None, True),
    ('0.5', r'\frac{1}{2}', True),
]
for a, b, expected in math_cases:
    ours = is_equiv(a, b)
    orig = math_equivalence.is_equiv(a, b)
    assert ours == orig == expected, f'MATH mismatch: {a!r} vs {b!r}'
print(f'MATH верификация: {len(math_cases)}/{len(math_cases)} совпадений')

print()
print('ВЕРИФИКАЦИЯ ПРОЙДЕНА: наши функции совпадают с оригиналами')

GSM8K extraction: 6/6
MATH extraction: 8/8
MATH is_equiv: 8/8
GSM8K comparison: 2/2
Metrics: mean=0.60, CI=[0.200, 1.000]

Тесты пройдены: 26/26
GSM8K верификация: regex совпадает с оригиналом
MATH верификация: 10/10 совпадений

ВЕРИФИКАЦИЯ ПРОЙДЕНА: наши функции совпадают с оригиналами


## 7. Сохранение parsing.py

Тесты и верификация пройдены — сохраняем функции в файл.  
Остальные ноутбуки подключают: `from parsing import ...`  

Файл автономный: не зависит от `grade_school_math` или `math_equivalence` —  
содержит весь нужный код внутри.

In [11]:
import inspect

# Собираем все функции для parsing.py
math_equiv_functions = [
    math_equivalence._fix_fracs,
    math_equivalence._fix_a_slash_b,
    math_equivalence._remove_right_units,
    math_equivalence._fix_sqrt,
    math_equivalence._strip_string,
    math_equivalence.is_equiv,
]

our_functions = [
    is_correct_gsm8k,
    last_boxed_only_string,
    remove_boxed,
    extract_answer_math,
    accuracy,
    bootstrap_ci,
]

header = '''"""Извлечение ответов из текста модели и сравнение с эталоном.

Код сравнения ответов — из оригинальных evaluation-скриптов:
    GSM8K:  openai/grade-school-math  (Cobbe et al., 2021)
    MATH:   hendrycks/math            (Hendrycks et al., 2021)

Верифицировано в 00_setup.ipynb: тесты + проверка совпадения
результатов с оригинальными пакетами.
"""

import re
import numpy as np

ANS_RE = re.compile(r"#### (\\-?[0-9\\.\\,]+)")


def extract_answer_gsm8k(text):
    \'\'\'Находит #### в тексте, возвращает число после него.
    Источник: openai/grade-school-math/dataset.py:extract_answer\'\'\'
    match = ANS_RE.search(text)
    if match:
        return match.group(1).strip().replace(",", "")
    return None
'''

# Записываем
path = '/content/drive/MyDrive/vkr_data/parsing.py'
with open(path, 'w') as f:
    f.write(header)
    for func in our_functions + math_equiv_functions:
        f.write('\n\n' + inspect.getsource(func))

# Проверяем
import sys
sys.path.insert(0, '/content/drive/MyDrive/vkr_data')
import parsing
import importlib
importlib.reload(parsing)

assert parsing.extract_answer_gsm8k('#### 42') == '42'
assert parsing.extract_answer_gsm8k('нет') is None
assert parsing.is_equiv(r'\frac{1}{2}', '0.5') == True
assert parsing.extract_answer_math(r'\boxed{42}') == '42'

print(f'parsing.py сохранён: {path}')
print(f'Размер: {len(open(path).read())} символов')

parsing.py сохранён: /content/drive/MyDrive/vkr_data/parsing.py
Размер: 7111 символов


/content/drive/MyDrive/vkr_data/parsing.py:204: SyntaxWarning: invalid escape sequence '\%'
  string = string.replace("\%", "")


## 8. Итог

In [12]:
print('Окружение настроено')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'Модель: {TARGET_MODEL}, revision: {target_sha}')
print(f'PRM: {PRM_MODEL}, revision: {prm_sha}')

print('parsing.py: верифицирован, сохранён на диск')


Окружение настроено
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
Модель: Qwen/Qwen2.5-1.5B, revision: 8faed761d45a263340a0528343f099c05c9a4323
PRM: Qwen/Qwen2.5-Math-PRM-7B, revision: 0610740060112df12585d00a1c5f4624d2f59051
parsing.py: верифицирован, сохранён на диск
